In [ ]:
# In[1]:

import json

# Load the task.json file
task_directory = 'rca100/cases/t045'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information
alert_window = task_data.get('alert_window', {})
alert_entity = task_data.get('alert_entity', {})
prompt_text = task_data.get('prompt_text', '')

alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert occurred for the operation `oteldemo.AdService/GetAds` in the `ad` service. The alert time window is from `2026-04-26T13:12:54.807167+00:00` to `2026-04-26T21:23:47+08:00`. The user requested help to analyze the root cause of the issue.

The original code execution output of IPython Kernel is also provided below for reference:

({'start': '2026-04-26T13:12:54.807167+00:00', 'end': '2026-04-26T21:23:47+08:00'}, {'entity_id': None, 'entity_name': 'ad::oteldemo.AdService/GetAds', 'entity_type': None, 'entity_domain': None}, '<code vibeops_object type="alert_event">ad错误次数告警<alert_event event_id="3a8a5c220c4e2a2abf46dae44177578d" trans_id="cufthm4pqd2iv9dpfa82khsbu5" rule_id="5ef8e2d3-5155-4a1f-8bc9-cff0c865d02e" rule_name="ad错误次数告警" entity_id="9052fc8c9077971fe3a1e1bf4bdbed5a" entity_name="oteldemo.AdService/GetAds" entity_type="apm.operation" entity_domain="apm" service="ad" alert_time="2026-04-26T21:16:47+08:00" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Extract all available KPIs and their corresponding components
available_kpis = metrics_data[['entity_set', 'entity_name', 'metric']].drop_duplicates()

available_kpis

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs across different components, including Kubernetes nodes (`k8s.node`), APM services (`apm.service.legacy`), APM operations (`apm.operation`), and APM instances (`apm.instance`). Key metrics include `node_ready_status`, `node_memory_usage_rate`, `node_cpu_usage_rate`, `error_rate`, and others. These KPIs can be analyzed to identify issues in the `ad` service and other potential root cause components.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                      entity_name                  metric
0          k8s.node                                        node_ready_status
1898       k8s.node                                    node_not_ready_status
4234       k8s.node                                   node_memory_usage_rate
6424       k8s.node                                   node_pod_running_count
8614       k8s.node                                      node_cpu_usage_rate
...             ...                              ...                     ...
96001  apm.instance        checkout-76dd8cf45b-fhsp2              error_rate
96023  apm.instance  frontend-proxy-67d64dc4f8-8c99b              error_rate
96073  apm.instance  product-catalog-6b8d76c6f-vbrb8              error_rate
96096  apm.instance             cart-ddddd7787-mjqhp              error_rate
96100  apm.instance  product-catalog-6b8d76c6f-zkf5z              error_rate

[372 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Create a new column to classify by 'component-KPI'
metrics_data['component_kpi'] = metrics_data['entity_name'] + '-' + metrics_data['metric']

# Group by 'component-KPI' and aggregate the time series
grouped_metrics = metrics_data.groupby('component_kpi')['value']

# Calculate the global P95 threshold for each 'component-KPI'
p95_thresholds = grouped_metrics.quantile(0.95).reset_index()
p95_thresholds.columns = ['component_kpi', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
The global P95 thresholds have been calculated for each 'component-KPI' time series in the `metrics.parquet` file. These thresholds represent the 95th percentile of metric values for each combination of component and KPI, providing a baseline to identify anomalies in subsequent analysis.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       7.000000
1                  -apiserver_non_2xx_read_request_qps       0.000000
2                 -apiserver_non_2xx_write_request_qps       0.443412
3                       -apiserver_requests_per_second      27.318604
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
367  shipping::oteldemo.ShippingService/GetQuote-av...       0.012090
368  shipping::oteldemo.ShippingService/GetQuote-re...   18989.500000
369  shipping::oteldemo.ShippingService/GetQuote-sl...     116.800000
370  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000013
371  shipping::oteldemo.ShippingService/ShipOrder-r...   18994.750000

[372 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-26T13:12:54.807167+00:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-26T21:23:47+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics data within the alert time window
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start_us) & (metrics_data['time'] <= alert_end_us)]

# Merge filtered metrics with P95 thresholds
merged_data = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the metric value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['p95_threshold']]

# Focus on anomalies related to the 'ad' service and other potential root cause components
ad_service_anomalies = anomalies[anomalies['entity_name'].str.contains('ad', case=False)]

ad_service_anomalies, anomalies

```
Out[4]:
```
During the alert time window, anomalies were identified in the `ad` service, specifically for the operation `ad::oteldemo.AdService/GetAds` and its associated instance `ad-59d75696db-lcsdf`. The metric `error_count` exceeded the global P95 threshold multiple times. Additionally, anomalies were observed in other components, such as high `node_memory_usage_rate` for Kubernetes nodes and elevated `workload` or `memory` metrics for services like `product-catalog`, `currency`, and `frontend-proxy`. These anomalies suggest potential issues in the `ad` service and other components that may contribute to the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                   time domain          entity_set                         entity_id                    entity_name  ...         value                            metric_set_id service                              component_kpi p95_threshold
8692   1777209294000000    apm       apm.operation  9052fc8c9077971fe3a1e1bf4bdbed5a  ad::oteldemo.AdService/GetAds  ...  3.262000e+03  apm@metric_set@apm.metric.apm.operation    None  ad::oteldemo.AdService/GetAds-error_count  3.260750e+03
8706   1777209714000000    apm       apm.operation  9052fc8c9077971fe3a1e1bf4bdbed5a  ad::oteldemo.AdService/GetAds  ...  3.301000e+03  apm@metric_set@apm.metric.apm.operation    None  ad::oteldemo.AdService/GetAds-error_count  3.260750e+03
10797  1777209294000000    apm        apm.instance  1a52bc421f2aa7bd270a0f4fc47a6fe7            ad-59d75696db-lcsdf  ...  3.262000e+03   apm@metric_set@apm.metric.apm.instance    None            ad-59d75696db-lcsdf-error_count  3.260750e+03
10811  1777209714000000    apm        apm.instance  1a52bc421f2aa7bd270a0f4fc47a6fe7            ad-59d75696db-lcsdf  ...  3.301000e+03   apm@metric_set@apm.metric.apm.instance    None            ad-59d75696db-lcsdf-error_count  3.260750e+03
12248  1777209264000000    apm  apm.service.legacy                                                               ad  ...  3.092000e+03                                     None    None                                   ad-error  2.930250e+03
12299  1777209264000000    apm  apm.service.legacy                                                   load-generator  ...  1.904789e+09                                     None    None                         load-generator-mem  1.904773e+09
12326  1777209294000000    apm  apm.service.legacy                                                               ad  ...  3.262000e+03                                     None    None                                   ad-error  2.930250e+03
12404  1777209324000000    apm  apm.service.legacy                                                               ad  ...  3.237000e+03                                     None    None                                   ad-error  2.930250e+03
12482  1777209354000000    apm  apm.service.legacy                                                               ad  ...  3.074000e+03                                     None    None                                   ad-error  2.930250e+03
13340  1777209684000000    apm  apm.service.legacy                                                               ad  ...  3.000000e+03                                     None    None                                   ad-error  2.930250e+03
13391  1777209684000000    apm  apm.service.legacy                                                   load-generator  ...  1.904822e+09                                     None    None                         load-generator-mem  1.904773e+09
13418  1777209714000000    apm  apm.service.legacy                                                               ad  ...  3.301000e+03                                     None    None                                   ad-error  2.930250e+03
13496  1777209744000000    apm  apm.service.legacy                                                               ad  ...  3.191000e+03                                     None    None                                   ad-error  2.930250e+03
13574  1777209774000000    apm  apm.service.legacy                                                               ad  ...  2.934000e+03                                     None    None                                   ad-error  2.930250e+03

[14 rows x 11 columns],                    time domain          entity_set                         entity_id      entity_name  ...         value                                     metric_set_id service             component_kpi p95_threshold
777    1777209204000000    k8s            k8s.node                                                     ...  9.572869e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_memory_usage_rate  9.423710e+01
778    1777209234000000    k8s            k8s.node                                                     ...  9.585889e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_memory_usage_rate  9.423710e+01
779    1777209264000000    k8s            k8s.node                                                     ...  9.594138e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_memory_usage_rate  9.423710e+01
780    1777209294000000    k8s            k8s.node                                                     ...  9.601638e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_memory_usage_rate  9.423710e+01
781    1777209324000000    k8s            k8s.node                                                     ...  9.608514e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   -node_memory_usage_rate  9.423710e+01
...                 ...    ...                 ...                               ...              ...  ...           ...                                               ...     ...                       ...           ...
13574  1777209774000000    apm  apm.service.legacy                                                 ad  ...  2.934000e+03                                              None    None                  ad-error  2.930250e+03
13634  1777209774000000    apm  apm.service.legacy                                    product-catalog  ...  5.693600e+04                                              None    None  product-catalog-workload  5.677225e+04
13670  1777209804000000    apm  apm.service.legacy                                           currency  ...  2.883584e+06                                              None    None              currency-mem  2.878464e+06
13685  1777209804000000    apm  apm.service.legacy                                     frontend-proxy  ...  6.481237e+07                                              None    None        frontend-proxy-mem  6.473660e+07
13761  1777209714000000    apm       apm.operation  28887014538b3aeed572d8e4810f6d88    frontend::GET  ...  3.006101e-01           apm@metric_set@apm.metric.apm.operation    None  frontend::GET-error_rate  2.546126e-01

[352 rows x 11 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', 'duration', and 'statusCode' are numeric for filtering and analysis
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')
traces_data['statusCode'] = pd.to_numeric(traces_data['statusCode'], errors='coerce')

# Convert alert time window to nanoseconds for filtering
alert_start_ns = alert_start.timestamp() * 1_000_000_000
alert_end_ns = alert_end.timestamp() * 1_000_000_000

# Filter traces within the alert time window
filtered_traces = traces_data[
    (traces_data['startTime'] >= alert_start_ns) & (traces_data['startTime'] <= alert_end_ns)
]

# Focus on traces related to the 'ad' service and its operation 'oteldemo.AdService/GetAds'
ad_traces = filtered_traces[
    (filtered_traces['serviceName'].str.contains('ad', case=False, na=False)) &
    (filtered_traces['spanName'].str.contains('oteldemo.AdService/GetAds', case=False, na=False))
]

# Identify the most downstream faulty service or container by analyzing statusCode, statusMessage, and duration
faulty_traces = ad_traces[(ad_traces['statusCode'] != 0) | (ad_traces['statusMessage'].notna())]
downstream_faults = faulty_traces.sort_values(by=['startTime', 'duration'], ascending=[True, False])

downstream_faults

```
Out[5]:
```
The traces related to the `ad` service and its operation `oteldemo.AdService/GetAds` during the alert time window reveal multiple spans with non-zero `statusCode` or error messages in the `statusMessage` field. These spans indicate faults in the `ad` service. The most downstream faulty spans in the call chain should be further analyzed to pinpoint the exact root cause, such as specific containers or external dependencies causing the issue.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                   spanName  ...                                         attributes  __topic__     __tag__:__pack_id__                                             events links
479823  325410d2c6a71f88216b5adaefa16b67  6a0c9bc0ef8357cc  3d49a851c621d36c    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-835F0  [{"attributes":{"message.type":"RECEIVED","mes...  None
479824  3485cf165b0d9ddd909ef959941f9140  c4f2d7f722c28e2a  b487fac2f22ef614    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-835F0  [{"attributes":{"message.type":"RECEIVED","mes...  None
479825  9df9b85a64dfe49538bfa6a1bd6be9bc  88526f24890777e1  3f830e6ae0aa7d6c    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-835F0  [{"attributes":{"message.type":"RECEIVED","mes...  None
479826  f2ca0fe3621e37ed3a80ece4ea695081  8632e39234e2a548  b0822f26d654c72e    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-835F0  [{"attributes":{"message.type":"RECEIVED","mes...  None
479827  f8c2894bdbae96c8ca385668b74f22b6  17d79c4a29abc3c6  f7e5b486d7ed009b    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-835F0  [{"attributes":{"message.type":"RECEIVED","mes...  None
...                                  ...               ...               ...  ...                        ...  ...                                                ...        ...                     ...                                                ...   ...
551825  f0361d034c4407f0ef25c5c9feb270b6  56282ad9abf49c51  90c798b3a9a99914    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-83613  [{"attributes":{"message.type":"RECEIVED","mes...  None
551826  8b06ddb539dc988d19453c18a691afc3  959a85adec9e4416  02d5588789aac1bb    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-83613  [{"attributes":{"message.type":"RECEIVED","mes...  None
551827  80731ff1d4265047b2671fa0e9def7a2  1e526bf3c95645c3  54e2833e541c9907    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-83613  [{"attributes":{"message.type":"RECEIVED","mes...  None
551829  065db7f2c2f62f1b69c3e83a4ca18bff  2dab08880fcd7ec1  df411d21360f9c63    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-83613  [{"attributes":{"message.type":"RECEIVED","mes...  None
551828  8ab52f90ab56d0a143de73e8d3d3df47  263374a670afe2d5  c7d92ff61c9886f3    2  oteldemo.AdService/GetAds  ...  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...             FCFFA032213EA9EA-83613  [{"attributes":{"message.type":"RECEIVED","mes...  None

[241 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start_us) & (logs_data['_time_'] <= alert_end_us)
]

# Focus on logs related to the 'ad' service and its operation 'oteldemo.AdService/GetAds'
ad_logs = filtered_logs[
    (filtered_logs['_container_name_'].str.contains('ad', case=False, na=False)) |
    (filtered_logs['content'].str.contains('oteldemo.AdService/GetAds', case=False, na=False))
]

# Look for error messages or patterns in the logs
error_logs = ad_logs[ad_logs['content'].str.contains('error|exception|fail', case=False, na=False)]

error_logs

```
Out[6]:
```
No error messages or relevant patterns were found in the logs related to the `ad` service or its operation `oteldemo.AdService/GetAds` during the alert time window. This suggests that the root cause may not be directly observable in the application logs. Further investigation into metrics, traces, or other components may be required.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Identify the most downstream faulty spans in the call chain for the `ad` service
# These are spans with non-zero statusCode or error messages in statusMessage
downstream_faulty_spans = faulty_traces.sort_values(by=['startTime', 'duration'], ascending=[True, False])

# Analyze the `attributes` and `events` fields of these spans for resource issues, dependency failures, or anomalies
downstream_faulty_spans_analysis = downstream_faulty_spans[['traceId', 'spanId', 'parentSpanId', 'spanName', 'serviceName', 'statusCode', 'statusMessage', 'duration', 'attributes', 'events']]

downstream_faulty_spans_analysis

```
Out[7]:
```
The most downstream faulty spans in the `ad` service's operation `oteldemo.AdService/GetAds` show non-zero `statusCode` values, indicating errors. The `attributes` and `events` fields of these spans contain details such as gRPC scope information and message events, but further inspection of these fields is needed to identify specific resource issues, dependency failures, or anomalies. These spans suggest that the root cause may involve gRPC communication or related dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId                   spanName serviceName  statusCode statusMessage  duration                                         attributes                                             events
479823  325410d2c6a71f88216b5adaefa16b67  6a0c9bc0ef8357cc  3d49a851c621d36c  oteldemo.AdService/GetAds          ad           2                 6376279  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
479824  3485cf165b0d9ddd909ef959941f9140  c4f2d7f722c28e2a  b487fac2f22ef614  oteldemo.AdService/GetAds          ad           2                 5308968  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
479825  9df9b85a64dfe49538bfa6a1bd6be9bc  88526f24890777e1  3f830e6ae0aa7d6c  oteldemo.AdService/GetAds          ad           2                 6505947  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
479826  f2ca0fe3621e37ed3a80ece4ea695081  8632e39234e2a548  b0822f26d654c72e  oteldemo.AdService/GetAds          ad           2                 3072469  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
479827  f8c2894bdbae96c8ca385668b74f22b6  17d79c4a29abc3c6  f7e5b486d7ed009b  oteldemo.AdService/GetAds          ad           2                 3181611  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
...                                  ...               ...               ...                        ...         ...         ...           ...       ...                                                ...                                                ...
551825  f0361d034c4407f0ef25c5c9feb270b6  56282ad9abf49c51  90c798b3a9a99914  oteldemo.AdService/GetAds          ad           2                 4150784  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
551826  8b06ddb539dc988d19453c18a691afc3  959a85adec9e4416  02d5588789aac1bb  oteldemo.AdService/GetAds          ad           2                 3883413  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
551827  80731ff1d4265047b2671fa0e9def7a2  1e526bf3c95645c3  54e2833e541c9907  oteldemo.AdService/GetAds          ad           2                 9412246  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
551829  065db7f2c2f62f1b69c3e83a4ca18bff  2dab08880fcd7ec1  df411d21360f9c63  oteldemo.AdService/GetAds          ad           2                 3695736  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...
551828  8ab52f90ab56d0a143de73e8d3d3df47  263374a670afe2d5  c7d92ff61c9886f3  oteldemo.AdService/GetAds          ad           2                 1045199  {"otel.scope.name":"io.opentelemetry.grpc-1.6"...  [{"attributes":{"message.type":"RECEIVED","mes...

[241 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```